# 🔑 Solutions: Lesson 0: Welcome & Refresher
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)

---
### Problem 0.1 &nbsp; ⭐ &nbsp; First look at the farm data

Load the raw farm records with `load_raw_farm()`. Print (a) the number of rows and columns, and (b) the names of all columns whose dtype is *text* (`object` or `str`).

In [2]:
raw = load_raw_farm()
print("rows, columns:", raw.shape)
text_cols = raw.select_dtypes(exclude="number").columns.tolist()
print("non-numeric columns:", text_cols)

rows, columns: (2485, 22)
non-numeric columns: ['farm_id', 'county', 'soil_type', 'experience', 'crop', 'season', 'planting_date', 'harvest_date', 'fertilizer_kg_ha', 'irrigation', 'pest_pressure']


**💬 Why this works:** `exclude='number'` is robust across pandas versions (`object` in pandas 2, `str` in pandas 3). Notice that dates and fertilizer are *text*: we will fix that in Lesson 1.

---
### Problem 0.2 &nbsp; ⭐ &nbsp; Spot the mess

Use `value_counts()` on the `crop` column of the raw data. How many *distinct spellings* are there? How many *real* crops should there be? (Do not clean anything yet, just observe.)

In [3]:
raw = load_raw_farm()
print(raw["crop"].value_counts())
print("\ndistinct spellings:", raw["crop"].nunique(), "| real crops: 5 (Maize, Beans, Wheat, Potato, Sorghum)")

crop
Maize           718
Beans           416
Potato          354
Wheat           220
Sorghum         214
maize            41
MAIZE            39
Maize            34
Irish potato     33
POTATO           32
 Maize           31
Bean             31
Wheat            30
beans            30
maize.           29
Corn             28
Beans            27
BEANS            27
SORGHUM          24
WHEAT            23
sorghum          23
wheat            23
Sorgum           20
potato           20
Potatoes         16
MAIZE             1
CORN              1
Name: count, dtype: int64

distinct spellings: 27 | real crops: 5 (Maize, Beans, Wheat, Potato, Sorghum)


**💬 Why this works:** Computers treat `'Maize'`, `'maize'` and `' Maize'` as three different crops. If we trained a model now, it would learn three separate 'maize' effects from fragmented data.

---
### Problem 0.3 &nbsp; ⭐⭐ &nbsp; Filter and create

From the raw data keep only farms with `farm_size_ha` greater than 5. Report how many there are, and add a new column `is_large` (True/False) to the *full* dataframe that marks farms bigger than 5 ha. What percentage of farms are large?

In [4]:
raw = load_raw_farm()
big = raw[raw["farm_size_ha"] > 5]
print("large farms:", len(big))
raw["is_large"] = raw["farm_size_ha"] > 5
print(f"{raw['is_large'].mean() * 100:.1f}% of rows are large farms")

large farms: 116
4.7% of rows are large farms


**💬 Why this works:** Comparisons produce boolean Series; `df[mask]` filters rows and `mean()` of booleans is a proportion. Note NaN > 5 is False, so missing sizes count as 'not large': a quiet assumption worth remembering.

---
### Problem 0.4 &nbsp; ⭐⭐ &nbsp; Group and summarise

Compute the **median** `rainfall_mm` for every `county` in the raw data and sort from wettest to driest. Why might the result look odd? (Look at how many counties you get!)

In [5]:
raw = load_raw_farm()
med = raw.groupby("county")["rainfall_mm"].median().sort_values(ascending=False)
print(med)
print("\nnumber of 'counties':", len(med), "(there are only 10 real ones!)")

county
kisii          1034.40
kisii           913.30
Kisii           892.30
kakamega        823.30
Kisii           819.75
KAKAMEGA        809.30
Kakamega        802.15
Bungoma         792.70
Bungoma         784.95
TransNzoia      761.00
bungoma         747.50
KIAMBU          746.30
UASIN GISHU     744.60
trans nzoia     725.65
Kiambu          716.80
kiambu          691.10
Trans Nzoia     684.15
NYERI           650.30
uasin gishu     648.20
nyeri           641.70
Uasin Gishu     639.60
Nyeri           638.30
Trans-Nzoia     636.40
Uasin-Gishu     632.70
Meru            614.40
MERU            613.85
UasinGishu      603.80
meru            536.80
trans-nzoia     516.80
Nakuru          510.40
Nakuru          488.30
NAKURU          478.00
nakuru          476.90
machakos        439.00
Machakos        433.00
Machakos        409.40
machakos        393.30
Name: rainfall_mm, dtype: float64

number of 'counties': 37 (there are only 10 real ones!)


**💬 Why this works:** Inconsistent spellings split one county into several groups, so any group statistic is wrong. Cleaning categories (Lesson 2) is a prerequisite for trustworthy aggregation.